In [22]:

import pandas as pd
reduced_df = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')

In [23]:
# Include only enagement service code 10469
filtered_df = reduced_df[reduced_df['Engagement Service Code'] == '10469']
# Include only engagements with a release date after January 1st, 2024
# filtered_df = filtered_df[filtered_df['Release Date'] > '2024-01-01']


In [24]:
# Read all sheets from the Excel file and consolidate into a single dataframe
# with a column indicating whether it's a navigate engagement based on tab name

nav_eng_analysis = pd.read_excel('./raw/navigate/EY Navigate engagements_v3_July2025.xlsx', sheet_name=None)
print("Available sheets:", list(nav_eng_analysis.keys()))

Available sheets: ['10469 Engagements', 'New Navigate', 'New Non-Nav', 'New Inactives', 'New Pivot', 'Mercury Data', 'Prior Analysis-->', 'Old Summary', 'Old Navigate', 'Old Non-Nav', 'Old Inactives']


In [25]:
# Consolidate all client/engagement sheets into a single dataframe with Navigate indicator
# Define which sheets contain Navigate vs Non-Navigate engagements
navigate_sheets = ['New Navigate', 'Old Navigate']
non_navigate_sheets = ['New Non-Nav', 'Old Non-Nav', 'New Inactives', 'Old Inactives']

# Common columns to extract from all sheets
common_columns = ['Client', 'Engagement', 'Engagement ID']

consolidated_rows = []

for sheet_name, df in nav_eng_analysis.items():
    # Only process Navigate and Non-Navigate sheets
    if sheet_name in navigate_sheets or sheet_name in non_navigate_sheets:
        # Filter to only rows that have an Engagement ID (skip totals/empty rows)
        if 'Engagement ID' in df.columns:
            sheet_df = df[df['Engagement ID'].notna()].copy()
            
            # Extract common columns only
            available_cols = [col for col in common_columns if col in sheet_df.columns]
            sheet_df = sheet_df[available_cols].copy()
            
            # Add Navigate indicator and source tracking
            sheet_df['is_navigate'] = sheet_name in navigate_sheets
            sheet_df['source_tab'] = sheet_name
            
            consolidated_rows.append(sheet_df)

# Combine all sheets into single dataframe
consolidated_df = pd.concat(consolidated_rows, ignore_index=True)

# Forward fill Client names (some sheets have client only on first row of group)
consolidated_df['Client'] = consolidated_df['Client'].ffill()

# Remove any remaining rows without engagement IDs
consolidated_df = consolidated_df.dropna(subset=['Engagement ID'])

# Add Client ID by merging with filtered_df
client_id_lookup = filtered_df[['Engagement ID', 'Client ID']].drop_duplicates()
consolidated_df = consolidated_df.merge(client_id_lookup, on='Engagement ID', how='left')

print(f"Consolidated {len(consolidated_df)} unique engagements")
print(f"\nNavigate breakdown:")
print(consolidated_df.groupby('is_navigate').size())
print(f"\nBy source tab:")
print(consolidated_df.groupby(['source_tab', 'is_navigate']).size())
print(f"\nClient ID match rate: {consolidated_df['Client ID'].notna().sum()}/{len(consolidated_df)}")

Consolidated 234 unique engagements

Navigate breakdown:
is_navigate
False    121
True     113
dtype: int64

By source tab:
source_tab     is_navigate
New Inactives  False           6
New Navigate   True           43
New Non-Nav    False          35
Old Inactives  False          43
Old Navigate   True           70
Old Non-Nav    False          37
dtype: int64

Client ID match rate: 204/234


In [26]:
# Merge filtered_df (all Mercury engagements) with consolidated_df (Navigate status)
# Create lookup from consolidated_df with Navigate status
navigate_lookup = consolidated_df[['Engagement ID', 'is_navigate', 'source_tab']].drop_duplicates(subset=['Engagement ID'])

# Merge Navigate status into filtered_df
nav_merged = filtered_df.copy()
nav_merged = nav_merged.merge(navigate_lookup, on='Engagement ID', how='left')

# Create a Navigate Status column that's more descriptive
def get_navigate_status(row):
    if pd.isna(row['is_navigate']):
        return 'Not Classified'
    elif row['is_navigate']:
        return 'Navigate'
    else:
        # Check if it's an inactive
        if 'Inactive' in str(row['source_tab']):
            return 'Inactive'
        return 'Non-Navigate'

nav_merged['Navigate Status'] = nav_merged.apply(get_navigate_status, axis=1)

# Summary stats
print(f"Total engagements in Mercury (10469): {len(nav_merged)}")
print(f"\nNavigate Status breakdown:")
print(nav_merged['Navigate Status'].value_counts())
print(f"\nFYTD Revenue by Navigate Status:")
print(nav_merged.groupby('Navigate Status')['ANSR / Tech Revenue FYTD'].sum().apply(lambda x: f'${x/1e6:,.2f}M'))

Total engagements in Mercury (10469): 176

Navigate Status breakdown:
Navigate Status
Navigate          67
Non-Navigate      46
Not Classified    36
Inactive          27
Name: count, dtype: int64

FYTD Revenue by Navigate Status:
Navigate Status
Inactive           $0.01M
Navigate           $4.15M
Non-Navigate       $1.37M
Not Classified    $11.06M
Name: ANSR / Tech Revenue FYTD, dtype: object


In [27]:
# Export to Excel with Key Metrics, Business Rules, and Raw tabs
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill

# Select columns to export
export_df = nav_merged[['Client ID', 'Client', 'Account Channel', 'Engagement ID', 
                        'Engagement Status', 'Engagement Service Code', 'Release Date',
                        'Engagement', 'ANSR / Tech Revenue ETD', 'ANSR / Tech Revenue FYTD',
                        'TER ETD', 'TER FYTD', 'Navigate Status']].copy()

# Generate timestamp and output path
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
output_path = f'./review/navigate/Navigate_Adoption_{timestamp}.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Write raw data first
    export_df.to_excel(writer, index=False, sheet_name='Raw')
    
    # Add Excel table formatting to raw sheet
    ws_raw = writer.sheets['Raw']
    table_range = f'A1:{get_column_letter(len(export_df.columns))}{len(export_df) + 1}'
    table = Table(displayName='NavigateEngagements', ref=table_range)
    style = TableStyleInfo(name='TableStyleMedium2', showFirstColumn=False,
                           showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    table.tableStyleInfo = style
    ws_raw.add_table(table)
    
    # Create Key Metrics sheet with Excel formulas
    wb = writer.book
    ws_metrics = wb.create_sheet('Key Metrics', 0)  # Insert at position 0 (first sheet)
    
    # Define styles
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    metric_font = Font(bold=True)
    
    # Set column widths
    ws_metrics.column_dimensions['A'].width = 40
    ws_metrics.column_dimensions['B'].width = 25
    
    # Header row
    ws_metrics['A1'] = 'Metric'
    ws_metrics['B1'] = 'Value'
    ws_metrics['A1'].fill = header_fill
    ws_metrics['B1'].fill = header_fill
    ws_metrics['A1'].font = header_font
    ws_metrics['B1'].font = header_font
    
    # Data row count (for formula references)
    data_rows = len(export_df) + 1  # +1 for header
    fytd_col = 'J'  # ANSR / Tech Revenue FYTD column
    status_col = 'M'  # Navigate Status column
    
    # Metric rows with Excel formulas
    metrics = [
        ('Total Engagements', f'=COUNTA(Raw!A2:A{data_rows})', None),
        ('Navigate Engagements', f'=COUNTIF(Raw!{status_col}2:{status_col}{data_rows},"Navigate")', None),
        ('Non-Navigate Engagements', f'=COUNTIF(Raw!{status_col}2:{status_col}{data_rows},"Non-Navigate")', None),
        ('Inactive Engagements', f'=COUNTIF(Raw!{status_col}2:{status_col}{data_rows},"Inactive")', None),
        ('Not Classified Engagements', f'=COUNTIF(Raw!{status_col}2:{status_col}{data_rows},"Not Classified")', None),
        ('% Engagements in Navigate', f'=B3/B2', '0.0%'),
        ('', '', None),  # Spacer row
        ('Total Revenue (FYTD)', f'=SUM(Raw!{fytd_col}2:{fytd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Navigate Revenue (FYTD)', f'=SUMIF(Raw!{status_col}2:{status_col}{data_rows},"Navigate",Raw!{fytd_col}2:{fytd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Non-Navigate Revenue (FYTD)', f'=SUMIF(Raw!{status_col}2:{status_col}{data_rows},"Non-Navigate",Raw!{fytd_col}2:{fytd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Inactive Revenue (FYTD)', f'=SUMIF(Raw!{status_col}2:{status_col}{data_rows},"Inactive",Raw!{fytd_col}2:{fytd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Not Classified Revenue (FYTD)', f'=SUMIF(Raw!{status_col}2:{status_col}{data_rows},"Not Classified",Raw!{fytd_col}2:{fytd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('% Revenue in Navigate', f'=B10/B9', '0.0%'),
    ]
    
    for i, (label, formula, num_format) in enumerate(metrics, start=2):
        ws_metrics.cell(row=i, column=1, value=label)
        if formula:
            ws_metrics.cell(row=i, column=2, value=formula)
        if num_format:
            ws_metrics.cell(row=i, column=2).number_format = num_format
        if label:  # Bold non-empty labels
            ws_metrics.cell(row=i, column=1).font = metric_font
    
    # Create Business Rules sheet
    ws_rules = wb.create_sheet('Business Rules', 1)  # Insert after Key Metrics
    
    # Set column widths
    ws_rules.column_dimensions['A'].width = 8
    ws_rules.column_dimensions['B'].width = 90
    
    # Header row
    ws_rules['A1'] = 'Rule #'
    ws_rules['B1'] = 'Business Rule'
    ws_rules['A1'].fill = header_fill
    ws_rules['B1'].fill = header_fill
    ws_rules['A1'].font = header_font
    ws_rules['B1'].font = header_font
    
    # Section header style
    section_fill = PatternFill(start_color='D9E2F3', end_color='D9E2F3', fill_type='solid')
    section_font = Font(bold=True)
    
    # Business rules content
    rules_content = [
        (None, 'EY Navigate Adoption Analysis - Filter Criteria', 'section'),
        ('1', 'Filter Engagement Service Code to include only 10469 (US Financial Planner Line)', 'rule'),
        (None, '', 'spacer'),
        (None, 'Navigate Classification', 'section'),
        (None, 'Navigate Status is determined by matching Engagement ID to the EY Navigate engagements spreadsheet', 'note'),
        (None, '"Navigate" - Engagement appears in New Navigate or Old Navigate tabs', 'note'),
        (None, '"Non-Navigate" - Engagement appears in New Non-Nav or Old Non-Nav tabs', 'note'),
        (None, '"Inactive" - Engagement appears in New Inactives or Old Inactives tabs', 'note'),
        (None, '"Not Classified" - Engagement not found in Navigate tracking spreadsheet', 'note'),
        (None, '', 'spacer'),
        (None, 'Data Sources', 'section'),
        (None, 'Mercury Data: reduced_coerced_engagement_list.feather (all 10469 engagements)', 'note'),
        (None, 'Navigate Status: EY Navigate engagements_v3_July2025.xlsx', 'note'),
        (None, '', 'spacer'),
        (None, 'Metrics Calculation', 'section'),
        (None, '% Engagements in Navigate = Navigate Engagements / Total Engagements', 'note'),
        (None, '% Revenue in Navigate = Navigate Revenue (FYTD) / Total Revenue (FYTD)', 'note'),
    ]
    
    for i, (rule_num, text, row_type) in enumerate(rules_content, start=2):
        if row_type == 'section':
            ws_rules.cell(row=i, column=1, value='')
            ws_rules.cell(row=i, column=2, value=text)
            ws_rules.cell(row=i, column=1).fill = section_fill
            ws_rules.cell(row=i, column=2).fill = section_fill
            ws_rules.cell(row=i, column=2).font = section_font
        elif row_type == 'rule':
            ws_rules.cell(row=i, column=1, value=rule_num)
            ws_rules.cell(row=i, column=2, value=text)
            ws_rules.cell(row=i, column=1).alignment = Alignment(horizontal='center')
        elif row_type == 'note':
            ws_rules.cell(row=i, column=1, value='')
            ws_rules.cell(row=i, column=2, value=text)

print(f'Saved to: {output_path}')
print(f'Raw data rows: {len(export_df)}')
print('Key Metrics sheet created with Excel formulas (using FYTD revenue)')
print('Business Rules sheet created')

Saved to: ./review/navigate/Navigate_Adoption_20260130_182015.xlsx
Raw data rows: 176
Key Metrics sheet created with Excel formulas (using FYTD revenue)
Business Rules sheet created
